This file generates some supplementary material (histograms and correlations) and also converts the LLM predictions into a dataframe with one entry for each student ("cta_with_schoolid2.csv"). This file is then analyzed in the R file "cta_paper_results.Rmd". A second dataframe, based on the ranking of the OOB predictions is also generated ("cta_with_schoolid_mod.csv"), which is also analyzed in that R file. 

In [1]:
import pandas as pd
import numpy as np
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy.stats import ttest_ind

In [2]:
def get_adj_risk_score():
    results = pd.read_csv("cta_results/pair_results_allpairs.csv")
    results['response'] = np.where((results['response'] == "Student 1") | (results['response'] == "Student 2"), results['response'], "Removed response")

    rating_counts_df = pd.concat([
    results.loc[results['response'] == 'Student 1', 'id.x'],
    results.loc[results['response'] == 'Student 2', 'id.y']
    ])

    rate_df = rating_counts_df.value_counts().reset_index()
    rate_df.columns = ['id', 'risk_score']

    unknown_counts_df = pd.concat([
    results.loc[results['response'] == 'Removed response', 'id.x'],
    results.loc[results['response'] == 'Removed response', 'id.y']
    ])

    unknown_df = unknown_counts_df.value_counts().reset_index()
    unknown_df.columns = ['id', 'unknown_count']

    cta = pd.read_csv("cta_preprocessed_unpaired_allpairs.csv") 
    results_df = pd.merge(pd.merge(cta, rate_df, on = 'id', how = 'left'), unknown_df, on = 'id', how = 'left')
    results_df['risk_score'] = results_df['risk_score'].fillna(0)
    results_df['unknown_count'] = results_df['unknown_count'].fillna(0)
    results_df['risk_score_adj'] = results_df['risk_score']/((results_df['pair_group_size']-1)*2 - results_df['unknown_count'])

    return results_df

In [3]:
results2 = get_adj_risk_score()
cta2 = pd.read_csv("cta_student_level_final.csv")
cta2 = cta2[['state', 'grdlvl', 'race', 'sex', 'spec_speced', 'spec_gifted', 'spec_esl', 'frl', 'y_yirt' ,'xirt', 'treatment', 'schoolidn', 'pair']]
cta2['id'] = range(1, len(cta2) + 1)

results2 = results2[['id', 'oob_preds', 'pair_group', 'pair_group_size', 'risk_score', 'risk_score_adj']]

merged_results = results2.merge(cta2, on = 'id', how = 'left')

merged_results['race'] = merged_results['race'].fillna("Unknown")
merged_results['sex'] = merged_results['sex'].fillna("Unknown")
merged_results['spec_speced'] = merged_results['spec_speced'].fillna("Unknown")
merged_results['spec_gifted'] = merged_results['spec_gifted'].fillna("Unknown")
merged_results['spec_esl'] = merged_results['spec_esl'].fillna("Unknown")
merged_results['frl'] = merged_results['frl'].fillna("Unknown")

# for numeric, mean impute and add another column that captures the missing values 
merged_results["xirt_mis"] = merged_results["xirt"].isna().astype(int)
merged_results['xirt'] = merged_results['xirt'].fillna(merged_results['xirt'].mean())


merged_results.to_csv("cta_with_schoolid2.csv") # this version is read into R for estimates 

## Supplementary Material

### Figure 5

In [ ]:
print(merged_results['risk_score_adj'].mean())
print(merged_results['risk_score_adj'].std())

In [ ]:
merged_results['risk_score_adj'].corr(merged_results['y_yirt'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(merged_results['risk_score_adj'], bins=20)

plt.xlabel('Adjusted Pair Score')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig("cta_pairscore_hist.png", dpi=300, bbox_inches="tight")
plt.savefig("cta_pairscore_hist.jpg", bbox_inches="tight", dpi = 600)
plt.show()

### Correlations (Table 13)

In [4]:
merged_results['tx'] = np.where(merged_results['state'] == "TX", 1, 0)
merged_results['ky'] = np.where(merged_results['state'] == "KY", 1, 0)
merged_results['la'] = np.where(merged_results['state'] == "LA", 1, 0)
merged_results['mi'] = np.where(merged_results['state'] == "MI", 1, 0)
merged_results['ct'] = np.where(merged_results['state'] == "CT", 1, 0)
merged_results['al'] = np.where(merged_results['state'] == "AL", 1, 0)
merged_results['nj'] = np.where(merged_results['state'] == "NJ", 1, 0)

for state in ['tx', 'ky', 'la', 'mi', 'ct', 'al', 'nj']:
    print(merged_results['risk_score_adj'].corr(merged_results[state]))

-0.1978609212130066
0.13945742692379307
0.08577690718912553
-0.04711324084775298
0.026890261403985694
0.05319693084474734
-0.019031737766328743


In [5]:
without_race = merged_results[merged_results['race'] != "Unknown"]
without_race['race'].value_counts()

without_race['white'] = np.where(without_race['race'] == 'WHITE NON-HISPANIC', 1, 0)
without_race['black'] = np.where(without_race['race'] == 'BLACK NON-HISPANIC', 1, 0)
without_race['hispanic'] = np.where(without_race['race'] == 'HISPANIC', 1, 0)
without_race['asian'] = np.where(without_race['race'] == 'ASIAN / PACIFIC ISLANDER', 1, 0)
without_race['other'] = np.where(without_race['race'] == 'OTHER RACE / MULTI-RACIAL', 1, 0)
without_race['american_indian'] = np.where(without_race['race'] == 'AMERICAN INDIAN / ALASKAN NATIVE', 1, 0)

for col in ['white', 'black', 'hispanic', 'asian', 'other', 'american_indian']:
    print(without_race['risk_score_adj'].corr(without_race[col]))

0.149177442423507
-0.09877164008420347
-0.07021089565332107
-0.01652937193164906
-0.005865974765127768
-0.004861427354214844


/tmp/ipykernel_1890417/2263079659.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  without_race['white'] = np.where(without_race['race'] == 'WHITE NON-HISPANIC', 1, 0)
/tmp/ipykernel_1890417/2263079659.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  without_race['black'] = np.where(without_race['race'] == 'BLACK NON-HISPANIC', 1, 0)
/tmp/ipykernel_1890417/2263079659.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer

In [6]:
merged_results['grdlvl_mod'] = np.where(merged_results['grdlvl'] =="H", 0, 1)


print(merged_results['risk_score_adj'].corr(merged_results['grdlvl_mod']))

merged_results['sex_mod'] = np.where(merged_results['sex'] =="M", 1, 0)


print(merged_results['risk_score_adj'].corr(merged_results['sex_mod']))

for col in ['spec_speced', 'spec_gifted', 'spec_esl', 'frl']:
    dropped_df = merged_results[merged_results[col] != "Unknown"]
    print(dropped_df['risk_score_adj'].corr(dropped_df[col]))

print(merged_results['risk_score_adj'].corr(merged_results['xirt']))

-0.04611042423610103
-0.02093351852111165
-0.134174444416263
0.07037066515999546
-0.08087768663645516
-0.002310300400946182
0.48085410778484144


## Experiment with Ranking Based Indicators

In [ ]:
results = pd.read_csv("cta_results/pair_results_allpairs.csv")
results['response_mod'] = np.where(results['oob_preds.x'] > results['oob_preds.y'], "Student 1", "Student 2")

In [ ]:
def get_adj_risk_score_mod():

    rating_counts_df = pd.concat([
    results.loc[results['response_mod'] == 'Student 1', 'id.x'],
    results.loc[results['response_mod'] == 'Student 2', 'id.y']
    ])

    rate_df = rating_counts_df.value_counts().reset_index()
    rate_df.columns = ['id', 'risk_score_mod']

    unknown_counts_df = pd.concat([
    results.loc[results['response_mod'] == 'Removed response', 'id.x'],
    results.loc[results['response_mod'] == 'Removed response', 'id.y']
    ])

    unknown_df = unknown_counts_df.value_counts().reset_index()
    unknown_df.columns = ['id', 'unknown_count']

    cta = pd.read_csv("cta_preprocessed_unpaired_allpairs.csv") 
    results_df = pd.merge(pd.merge(cta, rate_df, on = 'id', how = 'left'), unknown_df, on = 'id', how = 'left')
    results_df['risk_score_mod'] = results_df['risk_score_mod'].fillna(0)
    results_df['unknown_count'] = results_df['unknown_count'].fillna(0)
    results_df['risk_score_adj_mod'] = results_df['risk_score_mod']/((results_df['pair_group_size']-1)*2 - results_df['unknown_count'])

    return results_df

In [ ]:
results_mod = get_adj_risk_score_mod()
cta = pd.read_csv("cta_student_level_final.csv")
cta = cta[['state', 'grdlvl', 'race', 'sex', 'spec_speced', 'spec_gifted', 'spec_esl', 'frl', 'y_yirt' ,'xirt', 'treatment', "schoolidn"]]
cta['id'] = range(1, len(cta) + 1)

results_mod = results_mod[['id', 'oob_preds', 'pair_group', 'pair_group_size', 'risk_score_mod', 'risk_score_adj_mod']]

merged_results = results_mod.merge(cta, on = 'id', how = 'left')

# for categorical variables, replace NAs with "unknown"
merged_results['race'] = merged_results['race'].fillna("Unknown")
merged_results['sex'] = merged_results['sex'].fillna("Unknown")
merged_results['spec_speced'] = merged_results['spec_speced'].fillna("Unknown")
merged_results['spec_gifted'] = merged_results['spec_gifted'].fillna("Unknown")
merged_results['spec_esl'] = merged_results['spec_esl'].fillna("Unknown")
merged_results['frl'] = merged_results['frl'].fillna("Unknown")

# for numeric, mean impute and add another column that captures the missing values 
merged_results["xirt_mis"] = merged_results["xirt"].isna().astype(int)
merged_results['xirt'] = merged_results['xirt'].fillna(merged_results['xirt'].mean())

X = merged_results[["state", "grdlvl", 'race', 'sex', 'spec_speced', 'spec_gifted', 'spec_esl', 'frl', 'xirt_mis', 'oob_preds', 'xirt', 'y_yirt', 'risk_score_adj_mod', 'treatment', 'schoolidn']]
X.to_csv("cta_with_schoolid_mod.csv") # this version is read into R 